# LMI — تدريب مستقل في Google Colab دون Google Drive
هذه نسخة تنفيذية مصححة. **لا تدّعي تدريب AraBERT مسبقاً**؛ يشترط GPU واتصال إنترنت لتنزيل الأوزان. ارفع `LMI_PROJECT_REPAIRED.zip` فقط من الحاسب عند مطالبتك. إذا أردت حفظ النماذج، نفّذ خلية التصدير قبل انتهاء جلسة Colab. جميع ملفات النتائج داخل الأرشيف الأصلي هي نتائج Silver وليست Gold.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, subprocess, sys
uploaded = files.upload()
archive = next((Path(k) for k in uploaded if k.endswith('.zip')), None)
assert archive is not None, 'ارفع LMI_PROJECT_REPAIRED.zip'
root = Path('/content/LMI_PROJECT')
with ZipFile(archive) as z:
    for m in z.infolist():
        target = (Path('/content') / m.filename).resolve()
        if not target.is_relative_to('/content'):
            raise ValueError(f'Unsafe archive member: {m.filename}')
    z.extractall('/content')
assert (root / 'src/common.py').exists()
os.chdir(root)
print('Project:', root)


In [ ]:
def run(*args):
    subprocess.run(list(args),check=True,cwd='/content/LMI_PROJECT')
run(sys.executable,'-m','pip','install','-r','requirements.txt','-r','requirements_dl.txt')
import torch
assert torch.cuda.is_available(), 'تفعيل GPU من Runtime → Change runtime type ثم أعد تشغيل الخلية'
print('GPU:',torch.cuda.get_device_name(0))
run(sys.executable,'-m','scripts.verify_delivery')


## التدريب والتقييم
تُستخدم ملفات Train وValidation الفضية وتُحفظ الأوزان الناتجة في `models_dl`. التقييم على Test الفضية **ليس تقييماً على تسميات بشرية**. تشغّل الأوامر بفحص `check=True` كي لا تُخفى أخطاء التدريب.

In [ ]:
for task in ('opportunity_type','geographic_scope'):
    model_path=root/'models_dl/classification'/task
    weight_files = list(model_path.glob('*.safetensors')) + list(model_path.glob('pytorch_model*.bin'))
    if (model_path/'training_config.json').exists() and weight_files:
        print('Existing checkpoint:', task)
    else:
        run(sys.executable,'-m','src.training.train_classifier','--task',task)
    run(sys.executable,'-m','src.evaluation.evaluate_classifier','--task',task)
ner_path=root/'models_dl/ner'
if not ((ner_path/'training_config.json').exists() and (list(ner_path.glob('*.safetensors')) or list(ner_path.glob('pytorch_model*.bin')))):
    run(sys.executable,'-m','src.training.train_ner')
run(sys.executable,'-m','src.evaluation.evaluate_ner')


In [ ]:
run(sys.executable,'-m','pytest','-q','tests/test_repair_preflight.py','tests/test_dl_system.py')
run(sys.executable,'scripts/build_results_section.py')
print((root/'docs/RESULTS.md').read_text(encoding='utf8')[:7000])


## التطبيق
افتح العرض باستخدام منفذ Streamlit، أو استخدم رابط Tunnel عام عند الحاجة. لا تنشر بيانات خاصة أو رموز وصول في الرابط. تشغيل التطبيق لا يثبت الدقة على Gold.

In [ ]:
import subprocess
app_process=subprocess.Popen([sys.executable,'-m','streamlit','run','app_dl/main.py','--server.port','8501','--server.headless','true'],cwd=root)
print('Running on localhost:8501. Open Colab port preview if available.')


## تنزيل المشروع والنماذج بعد التدريب
احفظها قبل إيقاف Colab لأن مساحة العمل مؤقتة. الأرشيف يشمل الأوزان والتقارير مع المشروع.

In [ ]:
from google.colab import files
from shutil import make_archive
import shutil
archive_path=shutil.make_archive('/content/LMI_TRAINED_RESULT','zip',root_dir='/content',base_dir='LMI_PROJECT')
files.download(archive_path)
